# Py-HLA-Match · Demo Validation Notebook  
For research use only | Fraunhofer-Gesellschaft zur Förderung der angewandten Forschung e.V. 

⚠️ Disclaimer  

Py-HLA-Match is not certified or conformity assessed as a medical device software or in-vitro medical device software and is intended for research use only. It must therefore not be used for diagnosis or therapy of patients.

For more details on intended use, scope, and limitations, see the [Software Card](https://github.com/fraunhofer-izi/py-hla-match/blob/main/SOFTWARE_CARD.md).

> **Note:** The input file `cepph.csv` containing clinical HLA typing data is not publicly
> available due to GDPR. The results below were generated from that input and are published
> as `match_results_10.csv` and `match_results_all_full.csv` for independent verification
> of the concordance analysis. Access to the source data may be requested subject to ethics
> approval at Charité - Universitätsmedizin Berlin.

In [ ]:
from py_hla_match.parser import HLADataSource
from py_hla_match.export import PairwiseMatch

# define loci
standard_10_loci = ["A", "B", "C", "DRB1", "DQB1"]
standard_12_loci = ["A", "B", "C", "DRB1", "DQB1", "DPB1"]

In [2]:
data_path = "data/validation/cepph.csv"  # NOTE: not publicly available
output_path = "data/validation/match_results_10.csv"

# NOTE: we focus on clinically relevant standard 10 loci
# patient columns = 1-18, donor columns = 19-36 (row 0 is header)
# patient columns = 1-10, donor columns = 19-28 (row 0 is header)
src = HLADataSource(data_path, col_idx_start=2,  col_idx_stop=11, row_idx_start=1)
tgt = HLADataSource(data_path, col_idx_start=20, col_idx_stop=29, row_idx_start=1)

matching = PairwiseMatch(
    source=src,
    target=tgt,
    storage_filename=output_path,
    loci=standard_10_loci,
    overwrite=True,
)
matching.run()

Encountered invalid HLA Allele A*03:03 in row 111. Skipping Allele.
Unpaired allele A*01:01 in row 111.
Encountered invalid HLA Allele A*03:03 in row 111. Skipping Allele.
Unpaired allele A*01:01 in row 111.
HLA string 'DRB1*NE' at locus 'DRB1' is not a specific allele.
Encountered invalid HLA Allele C*03:02:02G in row 221. Skipping Allele.
Unpaired allele C*15:02:01G in row 221.
Locus C not found in donor data – matching will be reported as NOT_ASSESSABLE.
HLA string 'C*NA' at locus 'C' is not a specific allele.
Typing resolution insufficient for locus C (patient HLAPair(locus=C, hla1=C*03:02:02, hla2=C*15:02:01) / donor HLAPair(locus=C, hla1=C*NA, hla2=C*NA)).


In [3]:
matching.to_df()

,pair_index,A_1,A_2,B_1,B_2,C_1,C_2,DRB1_1,DRB1_2,DQB1_1,DQB1_2
0,0,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
1,1,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
2,2,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
3,3,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
4,4,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
...,...,...,...,...,...,...,...,...,...,...,...
408,408,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
409,409,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
410,410,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH
411,411,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH,ARD_MATCH


In [4]:
validation_data = pd.read_csv(data_path)
validation_data[['DonorTyp']]

matching_df = matching.to_df()

# create validation DataFrame with match counts
validation_df = pd.DataFrame({
    'donor_type': validation_data['DonorTyp'],
    'actual_matches': matching_df.apply(lambda row: sum('_MATCH' in str(val) for val in row), axis=1)
})

# filter for MUD and MMUD types and add expected matches
expected = {2.0: 10, 3.0: 9, 4.0: 8}
validation_df = validation_df[validation_df['donor_type'].isin([2.0, 3.0, 4.0])].copy()
validation_df['expected_matches'] = validation_df['donor_type'].map(expected)
validation_df['correct'] = validation_df['actual_matches'] == validation_df['expected_matches']

# incorrect predictions with detailed allele information
incorrect_rows = validation_df[~validation_df['correct']].index

In [5]:
# show results

print(f"Concordance: {validation_df['correct'].mean():.2%} ({validation_df['correct'].sum()}/{len(validation_df)})")
print(f"\nPotential Data Issues ({len(incorrect_rows)} rows):")

# define allele mapping
allele_cols = ['A_1', 'A_2', 'B_1', 'B_2', 'C_1', 'C_2', 'DRB1_1', 'DRB1_2', 'DQB1_1', 'DQB1_2']
patient_cols = ['patient-A*1', 'patient-A*2', 'patient-B*1', 'patient-B*2', 'patient-C*1', 'patient-C*2', 
               'patient-DRB1*1', 'patient-DRB1*2', 'patient-DQB1*1', 'patient-DQB1*2']
donor_cols = ['donor-A*1', 'donor-A*2', 'donor-B*1', 'donor-B*2', 'donor-C*1', 'donor-C*2',
             'donor-DRB1*1', 'donor-DRB1*2', 'donor-DQB1*1', 'donor-DQB1*2']

for row_id in incorrect_rows:
    expected = int(validation_df.loc[row_id, 'expected_matches'])
    actual = validation_df.loc[row_id, 'actual_matches']

    print(f"\nRow {row_id}: expected {expected}/10, actual {actual}/10")

    # patient row
    patient_alleles = [f"{validation_data.loc[row_id, col]:>5}" for col in patient_cols]
    # print(f"Patient:  {' | '.join(patient_alleles)}")

    # donor row  
    donor_alleles = [f"{validation_data.loc[row_id, col]:>5}" for col in donor_cols]
    # print(f"Donor:    {' | '.join(donor_alleles)}")

    # matching results - with NaN handling
    matching_results = []
    for col in allele_cols:
        val = matching_df.loc[row_id, col]
        if pd.isna(val):
            result = 'nan'
        elif '_MATCH' in str(val):
            result = 'match'
        else:
            result = 'mismatch'
        matching_results.append(f"{result:>8}")

    print(f"Matching: {' | '.join(matching_results)}")

Concordance: 98.75% (316/320)

Potential Data Issues (4 rows):

Row 111: expected 10/10, actual 8/10
Matching:      nan |      nan |    match |    match |    match |    match |    match |    match |    match |    match

Row 215: expected 9/10, actual 10/10
Matching:    match |    match |    match |    match |    match |    match |    match |    match |    match |    match

Row 221: expected 9/10, actual 7/10
Matching: mismatch |    match |    match |    match | mismatch | mismatch |    match |    match |    match |    match

Row 371: expected 10/10, actual 9/10
Matching:    match | mismatch |    match |    match |    match |    match |    match |    match |    match |    match


Clinician Feedback:
- Row 111: transcription error - allele "A*03:03" is incorrect
    - Developer Note: "A*03:03" is not a valid allele. This is treated as a missing allele, which results in "nan" for both, A_1 and A_2

- Row 215: transcription error - label "MMUD (9/10)" is incorrect

- Row 221: transcription error - allele "C*03:02:02G" is incorrect
    - Developer Note: "C*03:02:02G" is not a valid allele. This is treated as a missing allele, which results in "nan" for both, C_1 and C_2

- Row 371: transcription error - label "MUD (10/10)" is incorrect

In [6]:
# now run on all available loci and include detailed matching information
# NOTE: EBI API call to the DPB1 TCE model is included, which significantly increases processing time

data_path = "data/validation/cepph.csv"  # NOTE: not publicly available
output_path = "data/validation/match_results_all_full.csv"

# patient columns = 2-19, donor columns = 19-36 (row 0 is header)
src = HLADataSource(data_path, col_idx_start=2,  col_idx_stop=19, row_idx_start=1)
tgt = HLADataSource(data_path, col_idx_start=20, col_idx_stop=37, row_idx_start=1)

# loci
custom_loci = ["A", "B", "C", "DRB1", "DQB1", "DQA1", "DPB1", "DPA1", "DRBX"]

matching_full = PairwiseMatch(
    source=src,
    target=tgt,
    storage_filename=output_path,
    loci=custom_loci,
    overwrite=True,
    include_ard_details=True,
    include_molecular_details=True,
    include_dpb1_tce=True,
    include_homozygosity=True
)
matching_full.run()

HLA string 'DRBX*NE' at locus 'DRB345' is not a specific allele.
HLA string 'DQA1*01' at locus 'DQA1' is not a specific allele.
HLA string 'DPA1*01' at locus 'DPA1' is not a specific allele.
HLA string 'DQA1*05' at locus 'DQA1' is not a specific allele.
HLA string 'DPA1*02' at locus 'DPA1' is not a specific allele.
HLA string 'DQA1*03' at locus 'DQA1' is not a specific allele.
HLA string 'DQA1*NE' at locus 'DQA1' is not a specific allele.
HLA string 'DPA1*NE' at locus 'DPA1' is not a specific allele.
Encountered invalid HLA Allele DRB3*01:01:01 in row 62. Skipping Allele.
Unpaired allele DRB4*01:03:01 in row 62.
Encountered malformed HLA String DPA1*01NEW in row 82. Skipping Allele.
Unpaired allele DPA1*01:03:01 in row 82.
HLA string 'DPB1*NEW' at locus 'DPB1' is not a specific allele.
Encountered invalid HLA Allele A*03:03 in row 111. Skipping Allele.
HLA string 'DPB1*NE' at locus 'DPB1' is not a specific allele.
Unpaired allele A*01:01 in row 111.
Encountered malformed HLA String DQA

In [8]:
matching_full.to_df()

,pair_index,A_1,A_2,A_ard_1,A_ard_cert_1,A_ard_2,A_ard_cert_2,A_mol_1,A_mol_cert_1,A_mol_2,...,DRBX_ard_1,DRBX_ard_cert_1,DRBX_ard_2,DRBX_ard_cert_2,DRBX_mol_1,DRBX_mol_cert_1,DRBX_mol_2,DRBX_mol_cert_2,DRBX_homozygous_patient,DPB1_tce_status
0,0,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,NOT_ASSESSABLE,UNCERTAIN,NOT_ASSESSABLE,...,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,ARD Matched
1,1,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,NOT_ASSESSABLE,UNCERTAIN,NOT_ASSESSABLE,...,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,Non permissive GvH
2,2,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,NOT_ASSESSABLE,UNCERTAIN,NOT_ASSESSABLE,...,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,Permissive (core)
3,3,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,NOT_ASSESSABLE,UNCERTAIN,NOT_ASSESSABLE,...,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,Permissive (non-core GvH)
4,4,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,P_GROUP_MATCH,UNCERTAIN,NOT_ASSESSABLE,UNCERTAIN,NOT_ASSESSABLE,...,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NOT_APPLICABLE,NaN,Non permissive HvG
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
408,408,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,...,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,True,ARD Matched
409,409,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,...,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,True,Non permissive GvH
410,410,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,...,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,True,Non Permissive Bidirectional
411,411,ARD_MATCH,ARD_MATCH,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,...,G_GROUP_MATCH,CERTAIN,G_GROUP_MATCH,CERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,CODING_SEQUENCE_MATCH,UNCERTAIN,True,ARD Matched
